In [2]:
import pandas as pd
import numpy as np

# analysis window
START_DATE = pd.Timestamp("2023-12-29")
END_DATE   = pd.Timestamp("2025-12-31")
sec_px_csv = "sec_px.csv" # path to security price data
sec_metadata_csv = "sec_metadata.csv" # path to security metadata


In [3]:
def load_sec_px(path: str) -> pd.DataFrame:
    px = pd.read_csv(
        path,
        parse_dates=["date"],
        dayfirst=False
    )
    px = px.set_index("date").sort_index()
    px = px.apply(pd.to_numeric, errors="coerce")
    return px
sec_px = load_sec_px(sec_px_csv)

def restrict_window(px: pd.DataFrame,
                    start: pd.Timestamp,
                    end: pd.Timestamp) -> pd.DataFrame:
    return px.loc[(px.index >= start) & (px.index <= end)]
sec_px_window = restrict_window(sec_px, START_DATE, END_DATE)

def filter_eligible_securities(px: pd.DataFrame) -> pd.DataFrame:
    start_prices = px.iloc[0]
    end_prices   = px.iloc[-1]

    eligible = start_prices.notna() & end_prices.notna()
    px = px.loc[:, eligible]

    returns = px.pct_change()
    has_returns = returns.notna().sum() > 0

    return px.loc[:, has_returns]
sec_px_clean = filter_eligible_securities(sec_px_window)

In [4]:
def load_sec_metadata(path: str) -> pd.DataFrame:
    meta = pd.read_csv(path)
    meta.columns = meta.columns.str.lower()
    meta["date"] = pd.to_datetime(meta["date"])
    return meta
sec_metadata = load_sec_metadata(sec_metadata_csv)
def build_name_map(meta: pd.DataFrame) -> pd.Series:
    latest = (
        meta.sort_values("date")
            .groupby("sedol")
            .tail(1)
    )
    return latest.set_index("sedol")["name"]

name_map = build_name_map(sec_metadata)
name_map.head()

sedol
TS150329                               Far East Horizon (Det)
BD5CK8      CETC Cyberspace Security Technology Co. Ltd. C...
BD5CK3      Yuan Longping High-Tech Agriculture Co., Ltd. ...
BD5CKR                            SeeWay.ai Co., Ltd. Class A
BD5LY1                 Apeloa Pharmaceutical Co. Ltd. Class A
Name: name, dtype: object

In [5]:
def compute_annualised_twr(px: pd.DataFrame) -> pd.DataFrame:
    """
    Compute annualised Time-Weighted Return (TWR) for each security
    using monthly price data.

    Parameters
    ----------
    px : pd.DataFrame
        Cleaned price panel with DatetimeIndex and SEDOL columns

    Returns
    -------
    pd.DataFrame
        Columns: sedol, annualised_twr, n_months
    """
    # Monthly returns
    returns = px.pct_change()

    results = []

    for sedol in returns.columns:
        r = returns[sedol].dropna()

        if len(r) == 0:
            continue

        # Total time-weighted return
        twr_total = (1 + r).prod() - 1

        # Annualise based on number of months observed
        annualised_twr = (1 + twr_total) ** (12 / len(r)) - 1

        results.append({
            "sedol": sedol,
            "annualised_twr": annualised_twr,
            "n_months": len(r)
        })

    return pd.DataFrame(results)

twr_df = compute_annualised_twr(sec_px_clean)


In [6]:
twr_df["annualised_twr"].describe()
# twr_df["n_months"].describe()

# twr_df["n_months"].min(), twr_df["n_months"].max()

count    814.000000
mean       0.180755
std        0.332745
min       -0.436901
25%       -0.019264
50%        0.118526
75%        0.312294
max        2.946943
Name: annualised_twr, dtype: float64

In [7]:
def attach_security_names(twr_df: pd.DataFrame,
                          name_map: pd.Series) -> pd.DataFrame:
    out = twr_df.copy()
    out["name"] = out["sedol"].map(name_map)
    return out
twr_df_named = attach_security_names(twr_df, name_map)

In [8]:
# Get final result
def get_top_bottom_securities(twr_named: pd.DataFrame,
                              n: int = 3) -> tuple[pd.DataFrame, pd.DataFrame]:
    ranked = twr_named.dropna(subset=["name"])

    top = (
        ranked.sort_values("annualised_twr", ascending=False)
              .head(n)
              .loc[:, ["name", "annualised_twr"]]
    )

    bottom = (
        ranked.sort_values("annualised_twr", ascending=True)
              .head(n)
              .loc[:, ["name", "annualised_twr"]]
    )

    return top, bottom
top_securities, bottom_securities = get_top_bottom_securities(twr_df_named)
print("Top 3 Securities:")
print(top_securities)   
print("\nBottom 3 Securities:")
print(bottom_securities)

Top 3 Securities:
                                                  name  annualised_twr
325  Victory Giant Technology (HuiZhou) Co., Ltd. C...        2.946943
326            Eoptolink Technology Inc., Ltd. Class A        2.488145
504          Cambricon Technologies Corp. Ltd. Class A        2.152435

Bottom 3 Securities:
                                                  name  annualised_twr
198  Chongqing Zhifei Biological Products Co., Ltd....       -0.436901
484               Hygeia Healthcare Holdings Co., Ltd.       -0.413385
7                   Legend Biotech Corp. Sponsored ADR       -0.406499


## PART B 

In [21]:

def compute_sector_cumulative_return(sec_px, sec_metadata):

    # Move index to column
    px = sec_px.reset_index().rename(columns={'index': 'date'})

    # Reshape wide → long
    px_long = px.melt(
        id_vars='date',
        var_name='sedol',
        value_name='price'
    )

    # Ensure sorting
    px_long = px_long.sort_values(['sedol', 'date'])

    # Filter date range
    px_long = px_long[
        (px_long['date'] >= '2023-12-01') &
        (px_long['date'] <= '2025-12-31')
    ]

    # Compute returns
    px_long['return'] = (
        px_long.groupby('sedol')['price']
        .pct_change()
    )

    px_long = px_long.dropna(subset=['return'])

    # Merge sector metadata
    df = px_long.merge(
        sec_metadata[['sedol', 'sector']],
        on='sedol',
        how='left'
    )
    # Average returns at sector-date level
    sector_returns = (
        df.groupby(['sector', 'date'])['return']
        .mean()
        .reset_index()
    )

    # Convert to gross returns
    sector_returns['gross_return'] = 1 + sector_returns['return']

    # Compound once per sector
    sector_cum = (
        sector_returns.groupby('sector')['gross_return']
        .prod()
        .reset_index()
    )

    sector_cum['cumulative_return'] = sector_cum['gross_return'] - 1

    return sector_cum.sort_values('cumulative_return', ascending=False)


compute_sector_cumulative_return(sec_px, sec_metadata)
# sec_px.columns
# sec_metadata.head()

C:\Users\SP15548\AppData\Local\Temp\ipykernel_28784\2074245037.py:25: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


,sector,gross_return,cumulative_return
8,Materials,1.882192,0.882192
4,Financials,1.677760,0.677760
7,Information Technology,1.675821,0.675821
0,Communication Services,1.513550,0.513550
6,Industrials,1.507415,0.507415
1,Consumer Discretionary,1.438616,0.438616
3,Energy,1.320623,0.320623
10,Utilities,1.274135,0.274135
5,Health Care,1.042666,0.042666
2,Consumer Staples,1.036209,0.036209


In [22]:
def compute_sector_twr(twr_df: pd.DataFrame, metadata: pd.DataFrame) -> pd.DataFrame:
    """
    twr_df     : output from Part A — must have columns: sedol, annualised_twr, n_months
    metadata   : sec_metadata.csv loaded as DataFrame
    """
    # Get latest metadata row per SEDOL for sector mapping
    latest_meta = (
        metadata.sort_values("date")
                .groupby("sedol", as_index=False)
                .last()
                [["sedol", "name", "sector", "weight"]]
    )

    # Merge sector into twr results
    merged = twr_df.merge(latest_meta, on="sedol", how="left")

    # Group by sector: equal-weighted average annualised TWR
    sector_twr = (
        merged.groupby("sector")
              .agg(
                  avg_annualised_twr=("annualised_twr", "mean"),
                  num_securities=("sedol", "count")
              )
              .reset_index()
              .sort_values("avg_annualised_twr", ascending=False)
    )

    sector_twr["avg_annualised_twr_pct"] = (sector_twr["avg_annualised_twr"] * 100).round(2)

    return sector_twr


def display_sector_twr(sector_twr: pd.DataFrame) -> None:
    print(f"\n{'='*60}")
    print(f"  Sector Performance — Annualised TWR (Dec 2023 – Dec 2025)")
    print(f"{'='*60}")
    print(f"{'Rank':<6}{'Sector':<35}{'Avg TWR':>10}{'# Stocks':>10}")
    print(f"{'-'*60}")
    for rank, (_, row) in enumerate(sector_twr.iterrows(), start=1):
        print(f"{rank:<6}{row['sector']:<35}{row['avg_annualised_twr_pct']:>9.2f}%{int(row['num_securities']):>10}")
    print(f"{'='*60}")


# Run it
metadata = pd.read_csv("sec_metadata.csv")

sector_results = compute_sector_twr(twr_df, metadata)  # twr_df = eligible securities from Part A
display_sector_twr(sector_results)

# Export
sector_results[["sector", "avg_annualised_twr_pct", "num_securities"]].to_csv(
    "part_a_sector_results.csv", index=False
)
print("Sector CSV saved → part_a_sector_results.csv")



  Sector Performance — Annualised TWR (Dec 2023 – Dec 2025)
Rank  Sector                                Avg TWR  # Stocks
------------------------------------------------------------
1     Materials                              32.70%       107
2     Financials                             27.76%        99
3     Information Technology                 27.33%       142
4     Communication Services                 20.99%        32
5     Industrials                            17.43%       126
6     Consumer Discretionary                 14.86%        86
7     Energy                                 12.09%        25
8     Utilities                              10.36%        31
9     Health Care                            -0.03%        84
10    Real Estate                            -1.61%        25
11    Consumer Staples                       -2.47%        57
Sector CSV saved → part_a_sector_results.csv
